In [1]:
!git clone https://github.com/AnantaaKotal/Cluster-Medoid-Leakage-Attack.git


fatal: destination path 'Cluster-Medoid-Leakage-Attack' already exists and is not an empty directory.


In [2]:
%cd Cluster-Medoid-Leakage-Attack
!ls

/home/murilospalenza/Cluster-Medoid-Leakage-Attack
configs			examples			   README.md
diabetes.csv		gen_house.py			   requirements.txt
diabetes_syn_ctgan.csv	metrics_diabetes_ctgan_100ep.json
evaluate.py		metrics.json


In [3]:
%pip install -r requirements.txt


/home/murilospalenza/Cluster-Medoid-Leakage-Attack/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import sys
!{sys.executable} evaluate.py --dataset adult --real examples/adult_real.csv --syn examples/adult_synthetic_GReat.csv

/home/murilospalenza/Cluster-Medoid-Leakage-Attack/.venv/lib/python3.12/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")
✅ Done. Wrote: metrics.json
  mode: paper_defaults
  n_real / n_syn: 30162 / 29922
  n_medoids: 108
  ASR: {'0.01': 0.12037037037037036, '0.05': 0.5462962962962963, '0.1': 0.7870370370370371, '0.5': 0.9166666666666666}
  Coverage: {'0.01': 0.00046416020157814467, '0.05': 0.012233936741595385, '0.1': 0.05291426297990849, '0.5': 0.2340362044957231}
  dmin_stats: {'M': 108, 'min': 0.00047860293971534076, 'mean': 0.17658114003959072, 'median': 0.035097383854012155, 'max': 2.0064663066302244, 'p10': 0.006748224770855085, 'p90': 0.27714921455561164}


In [5]:
#BAIXAR DATASET + MODELO

In [6]:
import io, zipfile, shutil, subprocess, urllib.request
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm
from ucimlrepo import fetch_ucirepo
from sklearn.datasets import fetch_openml

BASE = Path.home() / "cmla_trabalho"
DADOS, MODELOS = BASE / "dados", BASE / "modelos"
DADOS.mkdir(parents=True, exist_ok=True)
MODELOS.mkdir(parents=True, exist_ok=True)


def baixar_com_barra(url, desc):
    resp = urllib.request.urlopen(url)
    total = int(resp.headers.get("Content-Length", 0)) or None
    buf = io.BytesIO()
    with tqdm(total=total, unit="B", unit_scale=True, desc=desc, leave=False) as barra:
        while True:
            bloco = resp.read(1 << 16)
            if not bloco:
                break
            buf.write(bloco)
            barra.update(len(bloco))
    buf.seek(0)
    return buf


def get_adult():
    REPO = Path.home() / "Cluster-Medoid-Leakage-Attack"
    shutil.copy(REPO / "examples" / "adult_real.csv", DADOS / "adult.csv")


def get_bank():
    url = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
    externo = zipfile.ZipFile(baixar_com_barra(url, "bank.zip"))
    interno = zipfile.ZipFile(io.BytesIO(externo.read("bank-additional.zip")))
    bank = pd.read_csv(interno.open("bank-additional/bank-additional-full.csv"), sep=";")
    bank.to_csv(DADOS / "bank.csv", index=False)


def get_telco():
    fetch_openml(data_id=42178, as_frame=True).frame.to_csv(DADOS / "telco.csv", index=False)


def get_drug():
    d = fetch_ucirepo(id=373)
    pd.concat([d.data.features, d.data.targets], axis=1).to_csv(DADOS / "drug.csv", index=False)


def clonar(nome, link):
    subprocess.run(
        ["git", "clone", "--depth", "1", link, str(MODELOS / nome)],
        check=True, capture_output=True,
    )


# (nome, arquivo/pasta de saída, função)
tarefas = [
    ("adult", DADOS / "adult.csv", get_adult),
    ("bank", DADOS / "bank.csv", get_bank),
    ("telco", DADOS / "telco.csv", get_telco),
    ("drug", DADOS / "drug.csv", get_drug),
    ("CTAB-GAN-Plus", MODELOS / "CTAB-GAN-Plus",
     lambda: clonar("CTAB-GAN-Plus", "https://github.com/Team-TUD/CTAB-GAN-Plus.git")),
    ("DPSDA (Tab-PE)", MODELOS / "DPSDA",
     lambda: clonar("DPSDA", "https://github.com/microsoft/DPSDA.git")),
]

with tqdm(total=len(tarefas), desc="Progresso geral") as geral:
    for nome, saida, funcao in tarefas:
        geral.set_postfix_str(nome)
        if saida.exists():
            tqdm.write(f"{nome}: já existe, pulando")
        else:
            funcao()
            tqdm.write(f"{nome}: ok")
        geral.update(1)

for nome in ["adult", "bank", "telco", "drug"]:
    print(nome, pd.read_csv(DADOS / f"{nome}.csv").shape)

Progresso geral:   0%|          | 0/6 [00:00<?, ?it/s]

adult: já existe, pulando
bank: já existe, pulando
telco: já existe, pulando
drug: já existe, pulando
CTAB-GAN-Plus: já existe, pulando
DPSDA (Tab-PE): já existe, pulando
adult (30162, 15)
bank (41188, 21)
telco (7043, 20)
drug (1885, 31)


In [7]:
from pathlib import Path
from tqdm.auto import tqdm

BASE = Path.home() / "cmla_trabalho"
PY_CTAB = BASE / ".venv-ctabgan/bin/python"
PY_TABPE = BASE / ".venv-tabpe/bin/python"
SCRIPT = BASE / "gerar_sinteticos.py"
SAIDA = BASE / "sinteticos"

EPOCHS_LIST = [150]       # artigo = padrão da biblioteca; os 300 já foram rodados
ITERS, EPSILONS = 30, [1.0]
N_SYN = 0                 # 0 = mesmo tamanho do real, como no artigo
datasets = ["drug", "telco", "adult", "bank"]
REFAZER = set()

jobs = []
for d in datasets:
    for ep in EPOCHS_LIST:
        jobs.append((f"{d}_ctabgan_ep{ep}", f"{d} | CTAB-GAN+ {ep} epochs",
                     f"{PY_CTAB} {SCRIPT} --model ctabgan --dataset {d} --epochs {ep}"))
    for eps in EPSILONS:
        nome = f"{d}_tabpe_eps{eps:g}" + (f"_n{N_SYN}" if N_SYN else "")
        jobs.append((nome, f"{d} | Tab-PE eps={eps:g}",
                     f"{PY_TABPE} {SCRIPT} --model tabpe --dataset {d} --epsilon {eps} --iters {ITERS} --n-syn {N_SYN}"))

resultado = {}
for arq, desc, cmd in tqdm(jobs, desc="Geração"):
    saida = SAIDA / f"{arq}.csv"
    if saida.exists() and arq not in REFAZER:
        resultado[arq] = "já existia, pulado"
        continue
    print(f"\n=== {desc} ===")
    get_ipython().system(cmd + (" --force" if arq in REFAZER else ""))
    resultado[arq] = "ok" if saida.exists() else "FALHOU"

print("\nResumo:")
for arq, status in resultado.items():
    print(f"  {arq}: {status}")

Geração:   0%|          | 0/8 [00:00<?, ?it/s]


=== drug | Tab-PE eps=1 ===
drug: (1885, 31) | cat=19 int=0 float=12 | label=vsa
/home/murilospalenza/cmla_trabalho/.venv-tabpe/lib/python3.12/site-packages/ot/datasets.py:204: SyntaxWarning: invalid escape sequence '\d'
  positive floating numbers corresponding to the isotropic variances in the principal subspace, for the source and target distributions, respectively. The same as \delta in :ref:`[1] <references-make_gauss-hd>`, Proposition 2.2
Tab-PE iterações:   0%|                                 | 0/30 [00:00<?, ?it/s]/home/murilospalenza/cmla_trabalho/modelos/DPSDA/pe/dp/gaussian.py:23: RuntimeWarning: overflow encountered in exp
  if np.isinf(np.exp(eps)):
fraction_per_label_id is not provided. Assuming the fraction of label ids in private data is public information.
Tab-PE iterações: 100%|████████████████████████| 30/30 [00:24<00:00,  1.21it/s]
OK: /home/murilospalenza/cmla_trabalho/sinteticos/drug_tabpe_eps1.csv (10000, 31) em 0.5 min

=== telco | CTAB-GAN+ ===
telco: (7032, 2